# Homework 10: Kubernetes and Model Serving

Machine Learning Zoomcamp 2026 — Module 10

Deploys the same lead-scoring API from Homework 5 to a local `kind` cluster. Q3, Q4, Q5 are Kubernetes concepts/given text, no cluster needed. Q6, Q7, Q8 come straight from the checked-in manifests. Q1 needed the actual model, so I reran it against the exact same artifact from Homework 5, confirming it's unchanged.

## Q1. Local container schema

This reuses the exact `05-deployment` artifact from Homework 5, same directory in the repo, same model. I confirmed `pipeline.bin` and `q6_test.py` are byte-for-byte identical to what I already verified in Homework 5, then ran the API and the client again to be sure rather than just assuming the old number still applies.

In [1]:
import hashlib

sha256 = hashlib.sha256(open("../homework-05/pipeline.bin", "rb").read()).hexdigest()
sha256, sha256 == "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"

('1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', True)

Ran `uvicorn predict:app` (same pinned environment as Homework 5: scikit-learn 1.7.2, numpy 2.3.3, fastapi 0.119.1, uvicorn 0.38.0) and the unmodified `q6_test.py` against it:

```
$ python q6_test.py
{'conversion_probability': 0.769799, 'conversion': True}
```

**Answer: 0.770** — identical to Homework 5's Q4/Q6, which makes sense since it's the same model, same code, just a different Docker tag (`2026-hw10` vs whatever tag was used before).

## Q2. Environment check

`kind --version` and `kubectl version --client` are local setup diagnostics, the homework says explicitly not to submit a version-specific answer. Run these yourself and note the output; they're about your machine, not the model.

## Q3. Kubernetes primitives

The smallest deployable unit Kubernetes creates and manages is a **Pod** (a Node is a machine, a Deployment manages a set of Pod replicas, a Service is networking, none of those are themselves "deployed" the way a Pod is).

**Answer: Pod**

## Q4. Default service type

Every Kubernetes cluster (`kind` included) auto-creates a `kubernetes` service in the `default` namespace pointing at the API server, and that service is always `ClusterIP` by default, it's how in-cluster workloads reach the API server without needing external exposure. This holds regardless of which machine or cloud the cluster runs on.

**Answer: ClusterIP**

## Q5. Load the local image

Directly from the homework's own command: `kind load docker-image zoomcamp-model:2026-hw10 --name mlzoomcamp-2026`.

**Answer: `kind load docker-image`**

## Q6. Container port

From the checked-in `deployment.yaml`:

In [2]:
print(open("deployment.yaml").read())

apiVersion: apps/v1
kind: Deployment
metadata:
  name: subscription
spec:
  selector:
    matchLabels:
      app: subscription
  replicas: 1
  template:
    metadata:
      labels:
        app: subscription
    spec:
      containers:
        - name: subscription
          image: zoomcamp-model:2026-hw10
          imagePullPolicy: IfNotPresent
          resources:
            requests:
              memory: "64Mi"
              cpu: "100m"
            limits:
              memory: "256Mi"
              cpu: "500m"
          ports:
            - containerPort: 9696
          readinessProbe:
            httpGet:
              path: /health
              port: 9696
            initialDelaySeconds: 2
            periodSeconds: 5



`containerPort: 9696` under the single container spec.

**Answer: 9696**

## Q7. Service selector

From the checked-in `service.yaml`:

In [3]:
print(open("service.yaml").read())

apiVersion: v1
kind: Service
metadata:
  name: subscription
spec:
  type: ClusterIP
  selector:
    app: subscription
  ports:
    - port: 80
      targetPort: 9696



`selector: app: subscription`, matching the label on the Deployment's Pod template.

**Answer: `app: subscription`**

## Q8. HPA configuration

From the checked-in `hpa.yaml`:

In [4]:
print(open("hpa.yaml").read())

apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: subscription-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: subscription
  minReplicas: 1
  maxReplicas: 3
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 20



`maxReplicas: 3` (with `minReplicas: 1`).

**Answer: 3**